# BayesTrade tutorial
Install the local package from the README. Run the following cells to verify a belief update, solve small decision problems, and audit a synthetic parent order.

## Bayes by hand
With prior (.6, .4) and observed likelihood (.8, .3), the unnormalized weights are (.48, .12).

In [ ]:
import numpy as np
from bayestrade.tabular import Beta, belief_update, backward_induction, pomdp_exact
b = belief_update([.6, .4], np.eye(2), [.8, .3])
assert np.allclose(b, [.8, .2])
print(b, Beta(2, 2).observe(1))


## A two-state execution MDP
State 0 is open and state 1 is filled. Action 0 waits and action 1 executes. The terminal charge for overdue inventory is 5. The second interval is cheaper.

In [ ]:
T = np.eye(2)[[[0, 1], [1, 1]]]
cost = np.array([[[0, 3], [0, 0]], [[0, 2], [0, 0]]])
values, policy = backward_induction(T, cost, [5, 0])
assert values[0, 0] == 2 and policy[0, 0] == 0
print(values, policy)


## The value of a sensing action
The third action costs 3 and reveals the state. The other two incur cost 10 in their mismatched state. With two decisions left, sensing first costs 3 in total.

In [ ]:
T = np.broadcast_to(np.eye(2)[:, None, :], (2, 3, 2))
O = [[[1, 0], [1, 0]], [[1, 0], [1, 0]], [[1, 0], [0, 1]]]
value, action = pomdp_exact(T, O, [[0, 10, 3], [10, 0, 3]], [0, 0], [.5, .5], 2)
assert (value, action) == (3, 2)
print(value, action)


## Same tape, different policies
The execution simulator hides regime. Inspect the booked fills, reconcile inventory, then recalculate shortfall.

In [ ]:
from bayestrade.market import SCENARIOS, execute, make_tape, shortfall
from bayestrade.policies import Baseline, BeliefPolicy, ObservedPolicy

scenario = SCENARIOS['switching']
tape = make_tape(scenario, 8, 78121)
for policy in (Baseline('twap', 6), BeliefPolicy(8, 6, 101)):
    outcome = execute(policy, tape, 6, scenario)
    fills = outcome['ledger']
    assert sum(fill.lots for fill in fills) == 6
    assert shortfall(fills, 6) == outcome['shortfall_bps']
    print(type(policy).__name__, round(outcome['shortfall_bps'], 2), fills)


## Learn unknown fill rates
In the observed-regime experiment, each standardized passive order contributes feedback to the Beta posterior of its own regime.

In [ ]:
scenario = SCENARIOS['unknown_fill']
tape = make_tape(scenario, 8, 78121)
agent = ObservedPolicy(6, learn=True, concentration=3)
print('Before:', agent.posterior)
outcome = execute(agent, tape, 6, scenario, observed=True)
print('After:', agent.posterior, 'shortfall:', outcome['shortfall_bps'])
